In [1]:
print("Hello World")

Hello World


In [2]:
! pip install -r requirements.txt

In [3]:
from peft import AutoPeftModelForTokenClassification
from transformers import AutoTokenizer, pipeline

c:\Users\ALEX\Downloads\skinny\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
checkpoint_path = r"C:\Users\ALEX\Downloads\skinny\backend\skinny k1\fine-tuning\results\checkpoint-5532"

In [5]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

In [6]:
model = AutoPeftModelForTokenClassification.from_pretrained(
    checkpoint_path,
    num_labels = 17
)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 19129.13it/s]
[transformers] DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 

In [7]:
id2label = {
    0: '0',
    1: 'B-ROLE', 2: 'I-ROLE',
    3: 'B-TECHNOLOGY', 4: 'I-TECHNOLOGY',
    5: 'B-ORGANIZATION', 6: 'I-ORGANIZATION',
    7: 'B-EXPERIENCE', 8: 'I-EXPERIENCE',
    9: 'B-SENIORITY', 10: 'I-SENIORITY',
    11: 'B-EDUCATION', 12: 'I-EDUCATION',
    13: 'B-LOCATION', 14: 'I-LOCATION',
    15: 'B-WORK_MODE', 16: 'I-WORK_MODE'
}

model.config.id2label = id2label
model.config.label2id = {v: k for k, v in id2label.items()}

In [8]:
ner_pipe = pipeline(
    "token-classification", 
    model=model, 
    tokenizer=tokenizer, 
    aggregation_strategy="simple"  # This groups sub-words into full words
)

In [9]:
text = """Job Title: AI/ML Engineering InternLocation: India (Remote)Employment Type: InternshipAbout Daice LabsDaice Labs, founded by MIT CSAIL scientists, is at the forefront of developing hybrid AI infrastructure designed for long-term, complex tasks. Our technology combines advanced AI models with neurosymbolic methods and bio-inspired system design to enable improved adaptability, verification, and context retention. We prioritize collaboration by empowering human teams to co-build and co-own outcomes while providing tools for governance, persistent context, and domain-specific environments.

Our innovative approach fosters adaptive learning, creating reusable building blocks for solutions across varied domains. Join us in shaping the future of governed, continuous-learning AI systems..About the RoleWe are hiring an AI/ML Engineering Intern for a 3-month internship (with potential for extension) to contribute to our research and engineering efforts remotely from India. You will work directly with senior engineers and researchers on active projects, gaining hands-on experience with production ML systems and hybrid AI architectures.ResponsibilitiesSupport the design and implementation of machine learning models under the guidance of senior engineersConduct experiments, analyze results, and document findingsAssist in building and maintaining data pipelines and preprocessing workflowsContribute to codebase improvements, testing, and documentationParticipate in team meetings, code reviews, and technical discussionsQualificationsCurrently pursuing or recently completed a Bachelor's or Master's degree in Computer Science, Data Science, AI, Machine Learning, or a related fieldFoundational knowledge of machine learning concepts, statistical modeling, and data structuresProficiency in Python and familiarity with ML libraries such as PyTorch or TensorFlowExperience with data preprocessing and dataset managementStrong analytical and problem-solving skillsClear written and verbal communication skills in EnglishAbility to work independently in a remote environment while meeting deadlinesPreferredPrior experience contributing to AI/ML research projects or open-source repositoriesFamiliarity with hybrid AI approaches, generative models, or symbolic AIExperience with version control (Git) and collaborative development workflowsCompensation: This is a paid internship.

Details will be discussed during the interview process.Duration: 3-6 months, with the possibility of extension based on mutual fit and project needs.Equal OpportunityDaice Labs is an equal opportunity employer. All qualified applicants will receive consideration for employment without regard to race, color, age, religion, sex, sexual orientation, gender identity or expression, national origin, disability, or any other characteristic protected under applicable law."""

In [10]:
result = ner_pipe(text)

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


In [11]:
import json

clean_results = []

for item in result:
    clean_results.append({
        "entity": item['entity_group'],
        "word": item['word'],
        "confidence": float(item['score']),  # This converts the numpy float to a standard float
        "start": item['start'],
        "end": item['end']
    })

# Print beautifully structured JSON with a 4-space indent
print(json.dumps(clean_results, indent=4))


[
    {
        "entity": "EDUCATION",
        "word": "Job",
        "confidence": 0.9887219667434692,
        "start": 0,
        "end": 3
    },
    {
        "entity": "EXPERIENCE",
        "word": "Title",
        "confidence": 0.996292769908905,
        "start": 3,
        "end": 9
    },
    {
        "entity": "ROLE",
        "word": ":",
        "confidence": 0.9993808269500732,
        "start": 9,
        "end": 10
    },
    {
        "entity": "ORGANIZATION",
        "word": "AI",
        "confidence": 1.0,
        "start": 10,
        "end": 13
    },
    {
        "entity": "ROLE",
        "word": "/ML Engineering Intern",
        "confidence": 0.9981870651245117,
        "start": 13,
        "end": 35
    },
    {
        "entity": "SENIORITY",
        "word": "Location",
        "confidence": 1.0,
        "start": 35,
        "end": 43
    },
    {
        "entity": "SENIORITY",
        "word": ":",
        "confidence": 0.5268105268478394,
        "start": 43,
        